In [0]:
from pyspark.sql.functions import col, coalesce, when, current_timestamp, lit

In [0]:
%python
try:
    files = dbutils.fs.ls("s3://nasa-exoplanet-datazuka/landing")
    for f in files:
        print(f.path)
except Exception as e:
    print(f"Erro detectado: {e}")

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS nasa_exoplanet.raw;


CREATE EXTERNAL VOLUME IF NOT EXISTS nasa_exoplanet.raw.nasa_data
LOCATION 's3://nasa-exoplanet-datazuka/landing/'
COMMENT 'Volume para os dados da NASA';

# Bronze

In [0]:
path = '/Volumes/nasa_exoplanet/raw/nasa_data/exoplanets.csv'
df = spark.read.csv(path, header=True, inferSchema=True)
display(df)

In [0]:
df.write.mode('overwrite').saveAsTable('nasa_exoplanet.bronze.exoplanet')

In [0]:
spark.read.table('nasa_exoplanet.bronze.exoplanet').display()

# SILVER


In [0]:
df_bronze = spark.read.table('nasa_exoplanet.bronze.exoplanet')
display(df_bronze)

In [0]:
#Selecionando os campos que podem servir para uma analise
df_silver = df_bronze['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 'koi_score', 'koi_period', 'koi_prad', 'koi_teq', 'koi_insol','koi_steff', 'koi_srad', 'koi_kepmag', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']
display(df_silver)

In [0]:
#renomeando as colunas


renamed_cols = {
    "kepid": "star_id",
    "kepoi_name": "koi_id",
    "kepler_name": "planet_name",
    "koi_disposition": "disposition",
    "koi_score": "confidence_score",
    "koi_period": "orbital_period_days",
    "koi_prad": "planet_radius_earth_ratio",
    "koi_teq": "temp_k",
    "koi_insol": "insolation_flux",
    "koi_steff": "stellar_temp_k",
    "koi_srad": "stellar_radius_solar_ratio",
    "koi_kepmag": "kepler_magnitude",
    "koi_fpflag_nt": "flag_not_transit_like",
    "koi_fpflag_ss": "flag_stellar_statistical",
    "koi_fpflag_co": "flag_centroid_offset",
    "koi_fpflag_ec": "flag_ephemeris_match"
}

df_silver = df_silver.select([col(old).alias(new) for old,new in renamed_cols.items()])
display(df_silver)


In [0]:
df_silver.count()

In [0]:
#Limpeza de dados: Aqui vamos eliminar todos os registros que sao FALSO POSITIVE

df_silver = df_silver.filter(df_silver.disposition != "FALSE POSITIVE")
df_silver.count()

In [0]:
display(df_silver)

In [0]:
##Existem planetas que ainda nao tem um nome porque ainda nao foram confirmados. Para nao deixa o campo null, vamos preencher esses valores com o koi_id
df_silver = df_silver.withColumn('planet_name', coalesce(col('planet_name'), col('koi_id'))) 
display(df_silver)


In [0]:
#Como n fazemos calculos com o valor da coluna de star_id, vamos mudar ela pra string

df_silver = df_silver.withColumn('star_id', col('star_id').cast('string'))
display(df_silver)

In [0]:
#Agora vamos usar as colunas de flag pra fazer uma coluna de filtro, dizendo se o planeta pode ser um FALSE POSITIVE, pois falhou em pelo menos uma das flags
df_silver = df_silver.withColumn('is_likely_false_positive', 
                                 when( 
                                      (col('flag_not_transit_like') == 1) | 
                                      (col('flag_stellar_statistical') == 1) | 
                                      (col('flag_centroid_offset') == 1) | 
                                      (col('flag_ephemeris_match') == 1), True
                                      ).otherwise(False)
)

display(df_silver)

In [0]:

#Agora vamos adicionar novas colunas com dados que podem ser interessantes para criar tabelas especificas na gold

df_silver = df_silver \
    .withColumn('temp_celsius', (col('temp_k') - 273.15)) \
    .withColumn('planet_type',
        when(col('planet_radius_earth_ratio') < 1.25, 'Terrestrial')
        .when(col('planet_radius_earth_ratio') < 2.0, 'Super-Earth')
        .when(col('planet_radius_earth_ratio') < 6.0, 'Neptunian')
        .otherwise('Gas Giant')) \
    .withColumn('is_habitable_candidate', 
        ((col('temp_celsius').between(-50, 50)) & (col('planet_radius_earth_ratio') < 2.0))) \
    .withColumn('ingested_at', current_timestamp()) 

display(df_silver)


In [0]:
cols_reorganizadas = [
    # 1. Identificadores 
    'star_id', 
    'koi_id', 
    'planet_name',
    
    # 2. Status e Classificação (É um planeta real?)
    'disposition', 
    'planet_type',
    'confidence_score',
    'is_habitable_candidate',
    'is_likely_false_positive',
    
    # 3. Flags de Erro/Qualidade
    'flag_not_transit_like', 
    'flag_stellar_statistical', 
    'flag_centroid_offset', 
    'flag_ephemeris_match',
    
    # 4. Dados do Planeta
    'orbital_period_days', 
    'planet_radius_earth_ratio', 
    'temp_k', 
    'temp_celsius',
    'insolation_flux',
    
    # 5. Dados da Estrela
    'stellar_temp_k', 
    'stellar_radius_solar_ratio', 
    'kepler_magnitude',
    
    # 6. Metadados do Sistema 
    'ingested_at'
]


df_silver_final = df_silver.select(*cols_reorganizadas)
display(df_silver_final)


In [0]:
df_silver_final.write.format("delta").mode("overwrite").saveAsTable("nasa_exoplanet.silver.exoplanets_cleaned")

In [0]:
spark.read.table("nasa_exoplanet.silver.exoplanets_cleaned").display()

# GOLD

In [0]:
df_gold = spark.read.table("nasa_exoplanet.silver.exoplanets_cleaned")
display(df_gold)

In [0]:
fact_exoplanet = df_gold.select('star_id', 'koi_id', 'disposition', 'orbital_period_days', 'planet_radius_earth_ratio', 'temp_celsius', 'confidence_score', 'ingested_at')
dim_planet = df_gold.select('koi_id', 'planet_name', 'planet_type', 'is_habitable_candidate').dropDuplicates(['koi_id'])
dim_star = df_gold.select('star_id', 'stellar_temp_k', 'stellar_radius_solar_ratio', 'kepler_magnitude').dropDuplicates(['star_id'])


In [0]:
dim_planet.write.format("delta").mode("overwrite").saveAsTable("nasa_exoplanet.gold.dim_planet")
dim_star.write.format("delta").mode("overwrite").saveAsTable("nasa_exoplanet.gold.dim_star")
fact_exoplanet.write.format("delta").mode("overwrite").saveAsTable("nasa_exoplanet.gold.fct_observations")

In [0]:
%sql
SELECT * FROM nasa_exoplanet.gold.dim_planet

In [0]:
%sql
SELECT * FROM nasa_exoplanet.gold.dim_star

In [0]:
%sql
SELECT * FROM nasa_exoplanet.gold.fct_observations